# 08 · Capstone: ShipFast and the research agent in Strands Agents

Notebooks 01 to 07 built the pieces. This notebook puts them together into the two agents every
framework in this project implements against the same contract (`contracts/`), and shows how they are
evaluated. The agents live in `strands_agents_impl/`, and `run_eval.py` runs the full evaluation with the same code.

**You will see**

- the ShipFast agent's design in Strands Agents, and why its Resolution is built in code
- two hard ShipFast scenarios run live through the **shared runner**: an approval, and a two-turn case
- one research question through the shared research runner
- this framework's full results, and all three frameworks side by side

In [1]:
# The agent module, its research workflow, and the adapter the shared runner drives.
import strands_agents_impl.shipfast as sf
from strands_agents_impl.research_flow import answer
from strands_agents_impl.shipfast import StrandsShipFast

# The shared harness from contracts/: the model check, the two runners and the test data.
from agent_contracts.harness import verify_model
from agent_contracts.harness.runners import run_question, run_scenario
from agent_contracts.research.scoring import load_questions
from agent_contracts.shipfast.scoring import load_scenarios

verify_model()  # stops with the fix if Ollama is not running or the shared model is missing
# Index the scenarios and questions by id, so a cell can pick one by name.
scenarios = {s["id"]: s for s in load_scenarios()}
questions = {q["id"]: q for q in load_questions()}

## The ShipFast agent

In Strands Agents the agent is **one agent per conversation** (it keeps its own messages), triage first as a native structured call
over the conversation, a fixed decline for out-of-scope messages, and the approval as a
`BeforeToolCallEvent` hook that interrupts before a refund over $100.

Every build fills in the scored `Resolution` the same way, with the shared
`agent_contracts.shipfast.resolution.derive_resolution` (DECISIONS.md 16): `intent` and `order_id` from
triage; `action`, `refund_amount` and `ticket_id` from what actually happened in the ledger and the
tickets; and only when a turn left no record does the model choose between `answered`, `needs_info` and
`declined`. The LangGraph and CrewAI component notebooks showed why: asked to fill in a Resolution
itself, the 8B model claimed refunds that never happened and invented ticket numbers. (In this
tutorial's notebook 03 the one Resolution asked for after the tools ran was right; one example is too few
to rely on, and the rule is the same in every build.)

In [2]:
# The module docstring in strands_agents_impl/shipfast.py describes the agent's design. It is shown here because
# the file is the one place that states the whole flow in a few lines.
print(sf.__doc__)

The ShipFast support agent in Strands Agents (ShipFast SPEC.md), the capstone of notebooks 01 to 07.

- One Strands `Agent` per customer conversation: it keeps the conversation itself (notebook 04).
- Triage first, as a native structured call over the conversation so far (shared `prompts.TRIAGE`);
  out-of-scope messages get a fixed decline and never reach the agent (policy P7).
- The agent has the shared tools and instructions, and the `RefundApproval` hook, which interrupts
  before an `issue_refund` over the shared $100 limit; `resume` answers the interrupt.

The Resolution is built by the shared `derive_resolution` from the ledger and tickets (DECISIONS.md 16);
the model only writes the reply and, when a turn left no record, picks the `Outcome`.



The docstring describes the agent. The method that does one customer turn is `send`, so its source shows the whole design: triage first, a fixed decline for out-of-scope messages, then the agent.

In [3]:
# send() is the adapter's one-turn method, in strands_agents_impl/shipfast.py. It is shown here because
# it is the whole agent in a dozen lines: triage, decline or agent call, then the Resolution.
import inspect

print(inspect.getsource(sf.StrandsShipFast.send))

    async def send(self, conv: dict, text: str, metrics: RunMetrics) -> Turn:
        conv["marker"], conv["hook"].metrics = TurnMarker.now(), metrics
        agent: Agent = conv["agent"]
        messages = conversation_messages(agent) + [{"role": "user", "content": [{"text": text}]}]
        conv["triage"] = structured_call(self.model, Triage, system=TRIAGE, metrics=metrics, messages=messages)
        if conv["triage"].intent == "other":
            agent.messages.extend([{"role": "user", "content": [{"text": text}]},
                                   {"role": "assistant", "content": [{"text": DECLINE}]}])
            return Turn(derive_resolution(conv["triage"], conv["marker"], DECLINE))
        result = await agent.invoke_async(text)
        return await self._finish(conv, result, metrics)



## Run through the shared runner

`run_scenario` is the same function every framework's `run_eval.py` uses. It resets the shared tools,
feeds each turn, resumes a paused approval with the scenario's decision, and checks the result against
`scenarios.json`. The framework only supplies the adapter: start a conversation, send a turn, resume.

`show()` turns a result into something you can read. It prints a summary block (verdict, cost and tools), a table with one row per customer turn (the `Resolution` the agent produced for that turn: intent, order, action and refund amount) with the start of the reply under it, and then any **problems**, which are the checks from `scenarios.json` that failed.

In [4]:
# show() prints one scenario result: a summary block, a table with one row per turn, then any problems.
def show(item):
    print(f"Scenario:    {item['id']}")
    print(f"Result:      {'PASS' if item['passed'] else 'FAIL'}")  # PASS means every check in scenarios.json held
    print(f"Cost:        {item['llm_calls']} model calls in {item['latency_s']:.0f}s")
    print(f"Tools used:  {', '.join(item['tool_calls']) or 'none'}")  # in the order the agent called them
    print()
    print(f"{'Turn':4}  {'Intent':15} {'Order':6} {'Action':11} {'Refund':>7}")
    for i, t in enumerate(item["turn_resolutions"], 1):
        refund = t["refund_amount"] or "-"  # "-" when no money moved
        print(f"{i:<4}  {t['intent']:15} {t['order_id'] or '-':6} {t['action']:11} {refund:>7}")
        print(f"      reply: {t['reply'].strip()[:160]}")  # the first 160 characters of what the customer saw
    print()
    if not item["problems"]:
        print("Problems:    none")
    for p in item["problems"]:
        print("Problem:    ", p)  # one check from scenarios.json that failed

First scenario: S3a, a late order that needs a refund above the approval limit.

In [5]:
# Scenario S3a: the late espresso machine, a $250 refund that needs approval.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(StrandsShipFast(), scenarios["S3a_late_refund_approved"])
show(result)

Scenario:    S3a_late_refund_approved
Result:      FAIL
Cost:        6 model calls in 50s
Tools used:  lookup_order, track_shipment, search_policy

Turn  Intent          Order  Action       Refund
1     refund_request  A1003  answered          -
      reply: The order A1003 is delayed, and the ETA is October 24, 2026. Since today is November 1, 2026, the order is more than 5 days past its ETA. According to the polic

Problem:     action: expected 'refunded', got 'answered'
Problem:     refund_amount: expected 250.0, got None


S3a is the late espresso machine, a $250 refund the supervisor approves. A passing run pauses at the
approval gate, is resumed with `approve`, and issues exactly $250.

In [6]:
# Scenario S4: the broken vase, a two-turn conversation.
# run_scenario plays the scenario's turns against a fresh agent and returns the scored result as a dict.
result = await run_scenario(StrandsShipFast(), scenarios["S4_damaged_needs_photo_then_refund"])
show(result)

Scenario:    S4_damaged_needs_photo_then_refund
Result:      PASS
Cost:        9 model calls in 90s
Tools used:  lookup_order, track_shipment, search_policy, issue_refund

Turn  Intent          Order  Action       Refund
1     complaint       A1004  needs_info        -
      reply: The vase set from your order A1004 arrived broken. According to our policy, you must provide a photo of the damage within 7 days of the delivery date. Since the
2     complaint       A1004  refunded       60.0
      reply: Your full refund of $60.00 for the broken vase set in order A1004 has been processed. You should see the refund reflected in your account within a few business 

Problems:    none


S4 is the broken vase: the first turn must ask for a photo and refund nothing (policy P3); the second,
which never names the order, must refund the $60 total. It needs memory and policy in the right order.

When we ran them, S4 passed: the first turn asked for the photo and refunded nothing, and the second,
which never names the order, refunded $60 once, using the conversation the agent keeps itself. S3a failed
the way it did in notebooks 03 and 06: the agent looked up the order, the tracking and the policy, told
the customer a full refund was due, and ended its turn without calling `issue_refund`. With no refund
attempt, the approval hook had nothing to interrupt, so there was no pause to approve.

## One research question

The research agent is the workflow from notebook 05, answered through the shared `run_question`. Q07 has
a stale article (9 provinces) beside the current one (14 provinces); mentioning the stale figure at all
fails the answer.

In [7]:
# Q07 has a stale article beside the current one. `answer` is this tutorial's research workflow, and
# run_question runs it on one question and returns the scored result as a dict.
item = await run_question(answer, questions["Q07_deployment"])

The result is a dict. Print its fields with a label each.

In [8]:
# Print the parts of the result worth reading, one labelled line each.
print("Passed:      ", item["passed"])  # True when the answer holds every expected fact and no stale one
print("Tools used:  ", ", ".join(item["tool_calls"]) or "none")
print("Model calls: ", item["llm_calls"])
print("Time:        ", f"{item['latency_s']:.0f}s")
print("Answer:      ", item["report"]["answer"])
print("Problems:    ", "; ".join(item["problems"]) or "none")  # what the scorer objected to, if anything

Passed:       False
Tools used:   web_search, web_search, web_search, web_search
Model calls:  8
Time:         110s
Answer:       Veltara Aquatech has deployed 2,800 TidePure units across 14 provinces by the end of 2025 [F04]. Earlier reports from 2023 indicated that TidePure units were operational in 9 provinces [F03].
Problems:     repeats stale or wrong figures: ['9 provinces']


## The full evaluation

`run_eval.py` runs every scenario and every question once and writes `results/shipfast/strands.json` and
`results/research/strands.json` (from the tutorial folder: `uv run python run_eval.py`). It takes about half an
hour on a 16 GB M4 Mac, so here we read the saved results instead of re-running them.

The table below has one row per scenario or question: its id, whether it passed, and how many seconds it took. Under a failing row, each `problem` is one check the scorer reported.

In [9]:
import json
from pathlib import Path

# The saved results live in results/<agent>/<framework>.json at the repo root, one level above the tutorial folder.
RESULTS = Path("..").resolve().parent / "results"


# Read one results file; returns None if that framework has not been run yet.
def load(agent, framework):
    path = RESULTS / agent / f"{framework}.json"
    return json.loads(path.read_text()) if path.exists() else None

In [10]:
# One block per agent: a summary line, then a table with one row per scenario or question.
for agent, key in (("shipfast", "scenarios"), ("research", "questions")):
    data = load(agent, "strands")
    summary = data["summary"]
    print(f"{agent}: {summary['passed']} of {summary['total']} passed (run at {data['run_at']})")
    print(f"  {'id':40} {'result':6} {'seconds':>7}")
    for it in data[key]:
        print(f"  {it['id']:40} {'PASS' if it['passed'] else 'FAIL':6} {it['latency_s']:7.0f}")
        for problem in it["problems"]:  # only failing items have problems
            print(f"      problem: {problem}")
    print()

shipfast: 4 of 8 passed (run at 2026-11-03T09:07:16+00:00)
  id                                       result seconds
  S1_status_with_followup                  PASS        56
  S2_lost_refund_auto                      PASS        34
  S3a_late_refund_approved                 FAIL        57
      problem: action: expected 'refunded', got 'answered'
      problem: refund_amount: expected 250.0, got None
  S3b_late_refund_rejected                 FAIL        52
      problem: action: expected 'escalated', got 'answered'
  S4_damaged_needs_photo_then_refund       PASS        78
  S5_outside_refund_window                 FAIL        35
      problem: action: expected 'declined', got 'refunded'
      problem: refund_amount: expected None, got 35.0
  S6_unknown_order                         FAIL        17
      problem: action: expected 'needs_info', got 'declined'
  S7_out_of_scope                          PASS         3

research: 9 of 10 passed (run at 2026-11-03T09:26:39+00:00)
  id      

## All three frameworks

The same contract, data, model, settings, tools, instructions, scoring and runner, so the differences
below come from the frameworks (and from how a small model reacts to each one's way of packaging the
same request).

In the table, `passed` is passed over total, `tokens` is the total tokens the model read and wrote, and `seconds` is the total run time, for that agent in that framework.

In [11]:
# Collect one row of totals per agent and framework, then print them as a table.
rows = []
for agent in ("shipfast", "research"):
    for framework in ("langgraph", "crewai", "strands"):
        data = load(agent, framework)
        if data:  # skip a framework whose results file does not exist yet
            s = data["summary"]
            rows.append((agent, framework, f"{s['passed']}/{s['total']}", s["total_tokens"], round(s["total_latency_s"])))

print(f"{'agent':9} {'framework':10} {'passed':7} {'tokens':>7} {'seconds':>8}")
for agent, framework, passed, tokens, seconds in rows:
    print(f"{agent:9} {framework:10} {passed:7} {tokens:7} {seconds:8}")

agent     framework  passed   tokens  seconds
shipfast  langgraph  7/8       29962      387
shipfast  crewai     7/8       34867      401
shipfast  strands    4/8       26981      331
research  langgraph  10/10     44361     1464
research  crewai     9/10      60243     1194
research  strands    9/10      48803     1147


Strands passed 4 of 8 ShipFast scenarios and 9 of 10 research questions. Two ShipFast failures are the
early stop above (S3a and S3b, the approval scenarios); S5 refunded the stand outside the 30-day window, as
LangGraph's run did; and in S6 the reply correctly asked the customer to check the order ID, but the
model's outcome choice labelled it `declined` rather than `needs_info`. The research failure, Q07, is the
one shown above: the answer mentioned the stale 9 provinces alongside the current 14.

The same model, instructions and tools ran under LangGraph and CrewAI, where the agent did attempt the
$250 refund and the gate paused it. So the early stop is something about how Strands packages the request
for this small model, worth investigating rather than tuning away, because it is exactly the kind of
difference this comparison exists to find. These are single runs on one laptop: a first measurement, not
a ranking.

## How the others do this

- **LangGraph** builds the same agent as a `StateGraph` with the specialist as a subgraph; the approval
  pause happens inside the tool call, and a checkpointer holds the conversation and the pause.
- **CrewAI** builds it as a **Flow**, one run per customer turn with the history in the flow's state; the
  approval pause sits between flow steps and resumes from saved state.

## Takeaways

- The contract keeps the comparison fair: one copy of the data, tools, instructions, scoring and runner.
- Code records the facts (refunds, tickets, pauses); the model writes the reply and makes one small call.
- Every framework could build both agents, but each needed its own shape for the approval pause and for
  memory, and each made different mistakes with the same model. Those differences are the comparison.